# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bitshank-2338/flyrank-ai-starter-notebook/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Provisional contract:** the daily fact grain is one client, content item and report date. Use March 2026 as an exploratory mid-panel month, verify uniqueness and date coverage with queries, and reserve June 2026 as a sealed outcome period. The eventual ranking unit is a content item at a decision date; aggregate only information available before that date. The target is later observed impression decline, not a product flag or contemporaneous outcome. A false positive costs an editor’s time; this output supports review, not an automatic change.

**Scope and limits:** use the daily performance fact, with dimensions only where necessary and joins validated for cardinality. Exclude label-derived future metrics, identifiers from published outputs, and unavailable source observations. Availability must be checked with `IS TRUE`; client histories are unbalanced and missing data are not zeros. Neither this sample nor a model proves that a refresh causes recovery.

In [2]:
%pip -q install duckdb
import duckdb
from google.colab import userdata
con = duckdb.connect()
token = userdata.get('HF_TOKEN')
con.execute("CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN ?)", [token])
source = "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')"
print('March coverage')
display(con.sql(f'SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day FROM {source}').df())
print('Duplicate grain groups (should be zero)')
display(con.sql(f'SELECT COUNT(*) AS duplicate_groups FROM (SELECT report_date, client_hash_id, content_hash_id FROM {source} GROUP BY 1,2,3 HAVING COUNT(*) > 1)').df())
print('Availability')
display(con.sql(f'SELECT COUNT(*) AS rows, COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available, COUNT(*) FILTER (WHERE ga4_data_available IS NOT TRUE) AS ga4_unavailable FROM {source}').df())

March coverage


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_day,last_day
0,9841378,2026-03-01,2026-03-31


Duplicate grain groups (should be zero)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_groups
0,0


Availability


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,ga4_available,ga4_unavailable
0,9841378,413966,9427412


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features (computed only before decision time):** trailing impressions, trailing clicks, click-through rate (clicks divided by impressions when positive), mean observed search position, and days of observed search history. Each uses past daily rows only; position and rates must respect availability and missingness.

**Label:** subsequent 30-day impressions below 80% of prior 30-day impressions, among items meeting a minimum prior exposure threshold. The future period cannot enter feature construction. **Context:** report date and pseudonymous client/content IDs for windows, joins and grouped evaluation, not predictors. **Excluded:** future impressions, outcome-period clicks, the label itself, derived decline flags, product recommendations, and query-table fields whose fixed window overlaps the outcome.

A deliberately leaky experiment would include future impressions and compare its held-out score with the past-only model, then remove it. No score is claimed until the comparison actually runs.

In [3]:
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
sample = pd.read_csv('https://raw.githubusercontent.com/Bitshank-2338/flyrank-ai-starter-notebook/main/data/raw/content_refresh_anonymized.csv')
y = sample['trend_direction'].eq('down').astype(int)
safe = ['impressions_90d','days_since_last_update','avg_position','ctr']
train, test = next(GroupShuffleSplit(n_splits=1, test_size=.25, random_state=42).split(sample, y, groups=sample['client_id']))
for name, cols in [('safe',safe), ('leaky',safe+['trend_pct'])]:
    X = sample[cols].fillna(0)
    model = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X.iloc[train],y.iloc[train])
    print(name, 'client-holdout accuracy:', round(accuracy_score(y.iloc[test],model.predict(X.iloc[test])),3))
print('Remove trend_pct: it determines the label and would not be available for a future outcome.')

safe client-holdout accuracy: 0.571
leaky client-holdout accuracy: 1.0
Remove trend_pct: it determines the label and would not be available for a future outcome.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

The executed March partition checks above returned **9,841,378 rows**, 2026-03-01 through 2026-03-31, and **zero duplicate** client/content/date groups. GA4 was available for **413,966** rows and unavailable or null for **9,427,412**. The large unavailable share makes a naive zero-filled engagement feature misleading. These counts apply to March only, not the entire panel.

The prototype uses the public starter snapshot for a separate leakage illustration; it is not a valid future-window warehouse evaluation. The genuine temporal label and held-out model remain to be constructed from non-overlapping warehouse windows.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

Client history starts differ; an absent analytics feed is not zero engagement. The fixed query snapshot overlaps the final outcome window, and must not be used as a feature without a prior-only alignment. Observational search metrics cannot establish whether an edit caused a change, explain user intent, or validate content quality. Evaluate with client and time separation before using the ranking prospectively.

In the starter-snapshot leakage illustration above, client-holdout accuracy was 0.571 using the selected non-label fields and 1.000 after adding `trend_pct`, which defines the proxy target. The latter is invalid and is excluded from the contract. Both scores are retrospective, not estimates of future performance; zero-filling was used only for this small diagnostic and should be replaced with explicit missingness handling in the actual model.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.